<a href="https://colab.research.google.com/github/sarah2005-cyber/auspex/blob/main/Preprocessing_steps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive._mount('/content/drive')


ValueError: mount failed

In [ ]:
!apt-get install ffmpeg

!ffmpeg -codecs | grep 729



Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --

**UNRAR THE ZIP FILE**

In [ ]:
!apt-get update -qq
!apt-get install -y unrar


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


In [ ]:
!unrar x \
"/content/drive/MyDrive/IIT - University of Westminster Material/4th Year/FYP/Implementation/Datasets/VStego800K dataset/train/g729a_0.rar" \
/content/extracted/

Streaming output truncated to the last 5000 lines.
Extracting  /content/extracted/g729a_0/English955.g729a                   98%  OK 
Extracting  /content/extracted/g729a_0/English9550.g729a                  98%  OK 
Extracting  /content/extracted/g729a_0/English95500.g729a                 98%  OK 
Extracting  /content/extracted/g729a_0/English95501.g729a                 98%  OK 
Extracting  /content/extracted/g729a_0/English95502.g729a                 98%  OK 
Extracting  /content/extracted/g729a_0/English95503.g729a                 98%  OK 
Extracting  /content/extracted/g729a_0/English95504.g729a                 98%  OK 
Extracting  /content/extracted/g729a_0/English95505.g729a                 98%  OK 
Extracting  /content/extracted/g729a_0/English95506.g729a                 98%  OK 
Extracting  /content/extracted/g729a_0/English95507.g729a                 98%  OK 
Extracting  /content/extracte

In [ ]:
import os
import shutil

SRC_COVER = "/content/extracted/g729a_0"
SRC_STEGO = "/content/extracted/g729a_Steg"

DST_COVER = "/content/selected/cover"
DST_STEGO = "/content/selected/stego"

TOTAL = 120_000
PER_LANG = TOTAL // 2

# os.makedirs(DST_COVER, exist_ok=True)
os.makedirs(DST_STEGO, exist_ok=True)

def select_files(src, prefix, n):
    return sorted([
        f for f in os.listdir(src)
        if f.startswith(prefix) and f.endswith(".g729a")
    ])[:n]

eng = select_files(SRC_COVER, "English", PER_LANG)
chi = select_files(SRC_COVER, "Chinese", PER_LANG)

selected = eng + chi
assert len(selected) == TOTAL

for f in selected:
    # shutil.copy(os.path.join(SRC_COVER, f), os.path.join(DST_COVER, f))
    shutil.copy(os.path.join(SRC_STEGO, f), os.path.join(DST_STEGO, f))

print(" Exactly 120k cover + 120k stego copied")


✅ Exactly 120k cover + 120k stego copied


In [ ]:
# !ls /content/selected/cover | wc -l
!ls "/content/drive/MyDrive/IIT - University of Westminster Material/4th Year/FYP/Implementation/Datasets/VStego800K dataset/decoded_wav/stego" | wc -l


KeyboardInterrupt: 

**CONVERTING .G729A AND PCM to .WAV**

**V1 (directly to GDrive)**

In [ ]:
import os
import subprocess

# Paths
input_cover = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/cover"
input_stego = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/stego"

output_base = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/decoded_wav"

output_cover = os.path.join(output_base, "cover")
output_stego = os.path.join(output_base, "stego")

os.makedirs(output_cover, exist_ok=True)
os.makedirs(output_stego, exist_ok=True)

def decode_pcm(inp, out):
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "s16le",
        "-ar", "8000",
        "-ac", "1",
        "-i", inp,
        "-ar", "44100",
        out
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def decode_g729(inp, out):
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "g729",
        "-i", inp,
        "-ar", "44100",
        "-ac", "1",
        out
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def decode_folder(src, dst):
    for f in os.listdir(src):
        ip = os.path.join(src, f)
        if not os.path.isfile(ip):
            continue

        op = os.path.join(dst, f.rsplit(".", 1)[0] + ".wav")

        # Skip if already exists
        if os.path.exists(op):
            continue

        if f.endswith(".pcm"):
            decode_pcm(ip, op)
        elif f.endswith(".g729a"):
            decode_g729(ip, op)


In [ ]:
# Decode the cover folder
# decode_folder(input_cover, output_cover)

# Decode the stego folder
decode_folder(input_stego, output_stego)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**V2 (decode the selected subset)**

In [ ]:
import os
import subprocess
from multiprocessing import Pool

# Paths
# input_cover = "/content/selected/cover"
input_stego = "/content/selected/stego"

output_base = "/content/decoded_wav"
# output_cover = os.path.join(output_base, "cover")
output_stego = os.path.join(output_base, "stego")

# os.makedirs(output_cover, exist_ok=True)
os.makedirs(output_stego, exist_ok=True)

# ---------- Decoders ----------

def decode_pcm(inp, out):
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "s16le",
        "-ar", "8000",
        "-ac", "1",
        "-i", inp,
        "-ar", "44100",
        out
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def decode_g729(inp, out):
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "g729",
        "-i", inp,
        "-ar", "44100",
        "-ac", "1",
        out
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# ---------- Worker ----------

def decode_worker(args):
    ip, op = args

    if os.path.exists(op):
        return

    if ip.endswith(".pcm"):
        decode_pcm(ip, op)
    elif ip.endswith(".g729a"):
        decode_g729(ip, op)

# ---------- Parallel folder decode ----------

def decode_folder_parallel(src, dst, workers=4):
    tasks = []

    for f in os.listdir(src):
        ip = os.path.join(src, f)
        if not os.path.isfile(ip):
            continue

        op = os.path.join(dst, f.rsplit(".", 1)[0] + ".wav")
        if not os.path.exists(op):
            tasks.append((ip, op))

    print(f"Decoding {len(tasks)} files from {src} using {workers} workers")

    with Pool(processes=workers) as pool:
        pool.map(decode_worker, tasks)

# ---------- Run ----------

if __name__ == "__main__":
    # decode_folder_parallel(input_cover, output_cover, workers=4)
    decode_folder_parallel(input_stego, output_stego, workers=4)


Decoding 120000 files from /content/selected/stego using 4 workers


**Move the decoded files to drive**

In [ ]:
import shutil
import os

src = "/content/local_standardized/stego"
dst = "/content/drive/MyDrive/standardized/stego"

os.makedirs(dst, exist_ok=True)

for f in os.listdir(src):
    shutil.move(
        os.path.join(src, f),
        os.path.join(dst, f)
    )


KeyboardInterrupt: 

**Standardize audio**

**Copy the decoded wav audio files into the /content directory**

In [ ]:
import os
import shutil
from multiprocessing import Pool

# Paths
SRC_STEGO = "/content/drive/MyDrive/IIT - University of Westminster Material/4th Year/FYP/Implementation/Datasets/VStego800K dataset/decoded_wav/cover"

DST_STEGO = "/content/local_decoded/cover"

os.makedirs(DST_STEGO, exist_ok=True)

BATCH_SIZE = 5000  # Copy 5000 files at a time
WORKERS = 4        # Number of parallel processes

def copy_file(args):
    src_file, dst_dir = args
    dst_file = os.path.join(dst_dir, os.path.basename(src_file))
    if not os.path.exists(dst_file):
        shutil.copy2(src_file, dst_file)

def copy_in_batches(src_dir, dst_dir, total_files, batch_size=BATCH_SIZE, workers=WORKERS):
    all_files = sorted([os.path.join(src_dir, f) for f in os.listdir(src_dir) if f.endswith(".wav")])[:total_files]
    print(f"Total files to copy from {src_dir}: {len(all_files)}")

    for i in range(0, len(all_files), batch_size):
        batch = all_files[i:i+batch_size]
        print(f"Copying files {i+1}-{i+len(batch)} ...")
        args = [(f, dst_dir) for f in batch]
        with Pool(workers) as pool:
            pool.map(copy_file, args)

# --- Copy cover and stego ---
TOTAL = 120000  # Number of files to copy per folder

copy_in_batches(SRC_STEGO, DST_STEGO, TOTAL)

print(" Copy complete: 120k stego files available locally")


Total files to copy from /content/drive/MyDrive/IIT - University of Westminster Material/4th Year/FYP/Implementation/Datasets/VStego800K dataset/decoded_wav/cover: 120000
Copying files 1-5000 ...
Copying files 5001-10000 ...
Copying files 10001-15000 ...
Copying files 15001-20000 ...
Copying files 20001-25000 ...
Copying files 25001-30000 ...
Copying files 30001-35000 ...
Copying files 35001-40000 ...
Copying files 40001-45000 ...
Copying files 45001-50000 ...
Copying files 50001-55000 ...
Copying files 55001-60000 ...
Copying files 60001-65000 ...
Copying files 65001-70000 ...
Copying files 70001-75000 ...
Copying files 75001-80000 ...
Copying files 80001-85000 ...
Copying files 85001-90000 ...
Copying files 90001-95000 ...
Copying files 95001-100000 ...
Copying files 100001-105000 ...
Copying files 105001-110000 ...
Copying files 110001-115000 ...
Copying files 115001-120000 ...
✅ Copy complete: 120k stego files available locally


**Find count commands**

In [ ]:
!find '/content/local_data/content/spec_512_spm/stego' -type f | wc -l


120000


In [ ]:
!unzip -l '/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/spm/spec_512_spm.zip' | tail -n +4 | wc -l


240005


**Standardize the audio**

In [ ]:
import soundfile as sf
import os
import numpy as np

TARGET_SR = 44100
TARGET_LEN = TARGET_SR * 1  # 1 second

def standardize_audio(input_path, output_dir, bit_depth="PCM_16", normalize=True):
    os.makedirs(output_dir, exist_ok=True)

    audio, sr = sf.read(input_path)

    # ---- Sanity check ----
    if sr != TARGET_SR:
        raise ValueError(f"{input_path} has sr={sr}, expected {TARGET_SR}")

    # ---- Ensure mono ----
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # ---- Enforce fixed length (1s) ----
    if len(audio) > TARGET_LEN:
        audio = audio[:TARGET_LEN]
    elif len(audio) < TARGET_LEN:
        pad = TARGET_LEN - len(audio)
        audio = np.pad(audio, (0, pad), mode="constant")

    # ---- Optional peak normalization ----
    if normalize:
        peak = np.max(np.abs(audio))
        if peak > 0:
            audio = audio / peak

    # ---- Write standardized file ----
    out_path = os.path.join(
        output_dir,
        os.path.basename(input_path).replace(".wav", "_standardized.wav")
    )

    sf.write(out_path, audio, TARGET_SR, subtype=bit_depth)
    return out_path


In [ ]:
input_dir = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/decoded_wav/stego"
std_dir = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/standardized/stego"

for fname in os.listdir(input_dir):
    if fname.endswith(".wav"):
        out_file = os.path.join(std_dir, fname.replace(".wav", "_standardized.wav"))
        if os.path.exists(out_file):
            continue  # Skip already processed files
        standardize_audio(os.path.join(input_dir, fname), std_dir)


In [ ]:
!cd /content
!zip -r standardized_cover.zip local_standardized/cover


Streaming output truncated to the last 5000 lines.
  adding: local_standardized/cover/English136544_standardized.wav (deflated 70%)
  adding: local_standardized/cover/English124881_standardized.wav (deflated 82%)
  adding: local_standardized/cover/Chinese32090_standardized.wav (deflated 18%)
  adding: local_standardized/cover/English140341_standardized.wav (deflated 21%)
  adding: local_standardized/cover/Chinese102955_standardized.wav (deflated 33%)
  adding: local_standardized/cover/Chinese22865_standardized.wav (deflated 56%)
  adding: local_standardized/cover/English127600_standardized.wav (deflated 77%)
  adding: local_standardized/cover/English141769_standardized.wav (deflated 29%)
  adding: local_standardized/cover/Chinese13448_standardized.wav (deflated 17%)
  adding: local_standardized/cover/Chinese18747_standardized.wav (deflated 66%)
  adding: local_standardized/cover/Chinese127517_standardized.wav (deflated 32%)
  adding: local_standardized/cover/Chinese19729_standardized.w

In [ ]:
!mv /content/standardized_cover.zip \
   "/content/drive/MyDrive"


In [ ]:
!unzip -l "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/standardized/standardized_stego.zip" \
| grep '\.wav$' | wc -l


120000


**MULTI WINDOW SPECTOGRAM STREAM PROCESSING**

**Extract the standardized files into content**

In [ ]:
!unzip "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/standardized/standardized_stego.zip" \
-d /content

Streaming output truncated to the last 5000 lines.
  inflating: /content/local_standardized/stego/English136544_standardized.wav  
  inflating: /content/local_standardized/stego/English124881_standardized.wav  
  inflating: /content/local_standardized/stego/Chinese32090_standardized.wav  
  inflating: /content/local_standardized/stego/English140341_standardized.wav  
  inflating: /content/local_standardized/stego/Chinese102955_standardized.wav  
  inflating: /content/local_standardized/stego/Chinese22865_standardized.wav  
  inflating: /content/local_standardized/stego/English127600_standardized.wav  
  inflating: /content/local_standardized/stego/English141769_standardized.wav  
  inflating: /content/local_standardized/stego/Chinese13448_standardized.wav  
  inflating: /content/local_standardized/stego/Chinese18747_standardized.wav  
  inflating: /content/local_standardized/stego/Chinese127517_standardized.wav  
  inflating: /content/local_standardized/stego/Chinese19729_standardized.

**Check for matching files in stego and cover**

In [ ]:
# Filenames must be identical
!comm -3 \
  <(ls /content/local_standardized/cover | sort) \
  <(ls /content/local_standardized/stego | sort)

In [ ]:
import os
import librosa
import numpy as np

# ====== Parameters ======
cover_dir = "/content/local_standardized/stego"
# stego_dir = "/content/drive/MyDrive/.../stego_standardized"
all_dirs = [cover_dir]

sr = 44100
n_fft_1024, hop_1024 = 1024, 512
n_fft_512, hop_512 = 512, 256

max_frames_1024 = 0
max_frames_512 = 0

# ====== Scan all files ======
for d in all_dirs:
    for fname in os.listdir(d):
        if not fname.endswith(".wav"):
            continue

        file_path = os.path.join(d, fname)
        audio, _ = librosa.load(file_path, sr=sr, mono=True)

        # STFT 1024
        spec1024 = librosa.stft(audio, n_fft=n_fft_1024, hop_length=hop_1024, window="hamming")
        frames1024 = spec1024.shape[1]
        if frames1024 > max_frames_1024:
            max_frames_1024 = frames1024

        # STFT 512
        spec512 = librosa.stft(audio, n_fft=n_fft_512, hop_length=hop_512, window="hamming")
        frames512 = spec512.shape[1]
        if frames512 > max_frames_512:
            max_frames_512 = frames512

print("Max frames (1024-window) =", max_frames_1024)
print("Max frames (512-window)  =", max_frames_512)


Max frames (1024-window) = 87
Max frames (512-window)  = 173


**Fixed parameter filtering**

In [ ]:
import os
import numpy as np
import librosa
from scipy.signal import convolve2d

# =========================
# Paths (EDIT THESE)
# =========================
INPUT_DIR = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/standardized/cover"

# OUT_SPEC1024 = "/content/spec_1024/cover"
# OUT_SPEC512  = "/content/spec_512/cover"

OUT_SPM1024  = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/spec_1024_spm/cover"
OUT_SPM512   = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/spec_512_spm/cover"

# os.makedirs(OUT_SPEC1024, exist_ok=True)
# os.makedirs(OUT_SPEC512,  exist_ok=True)
os.makedirs(OUT_SPM1024,  exist_ok=True)
os.makedirs(OUT_SPM512,   exist_ok=True)

# =========================
# Uniform time-frame targets (EDIT IF NEEDED)
# =========================
# These define the time dimension after padding/trimming
MAX_FRAMES_1024 = 87   # target time_frames for n_fft=1024 (hop=512)
MAX_FRAMES_512  = 173  # target time_frames for n_fft=512 (hop=256)

EPS = 1e-8

# =========================
# 1.3.1 + 1.3.2: Spectrogram (STFT + log + minmax + pad)
# =========================
def compute_log_norm_padded_spec(audio, sr, n_fft, hop_length, max_frames):
    """
    Returns spectrogram of shape (freq_bins, max_frames)
    - STFT with Hamming window
    - log10 magnitude
    - min-max normalization to [0,1]
    - zero-pad or trim along time axis to max_frames
    """
    # STFT
    S = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length, window="hamming")
    S_mag = np.abs(S)

    # Log magnitude
    S_log = np.log10(S_mag + EPS)

    # Min-max normalize per spectrogram
    s_min, s_max = S_log.min(), S_log.max()
    S_norm = (S_log - s_min) / (s_max - s_min + EPS)

    # Pad/trim along time axis
    freq_bins, time_frames = S_norm.shape
    if time_frames < max_frames:
        pad_t = max_frames - time_frames
        S_fixed = np.pad(S_norm, ((0, 0), (0, pad_t)), mode="constant")
    else:
        S_fixed = S_norm[:, :max_frames]

    return S_fixed  # (freq_bins, max_frames)

# =========================
# 1.3.3: Fixed-Parameter Spectrogram Filtering (SPM)
# =========================
# High-pass kernels
KERNEL_STRONG = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
], dtype=np.float32)

KERNEL_LIGHT = np.array([
    [ 0, -1,  0],
    [-1,  4, -1],
    [ 0, -1,  0]
], dtype=np.float32)

def apply_spm_filters(spec, kernels, weights=None):
    """
    Apply 3x3 high-pass kernels via 2D convolution (same padding),
    then combine (weighted sum), ReLU, and re-normalize to [0,1].
    """
    if weights is None:
        weights = [1.0] * len(kernels)

    filtered_sum = np.zeros_like(spec)
    for ker, w in zip(kernels, weights):
        # convolve2d over (freq, time)
        conv = convolve2d(spec, ker, mode="same", boundary="symm")
        filtered_sum += w * conv

    # Simple artifact enhancement: ReLU-like clamp
    filtered_sum = np.maximum(filtered_sum, 0.0)

    # Re-normalize to [0,1] (avoid all-zeros)
    fmin, fmax = filtered_sum.min(), filtered_sum.max()
    if fmax - fmin < EPS:
        return np.zeros_like(filtered_sum)
    return (filtered_sum - fmin) / (fmax - fmin + EPS)

# =========================
# Main loop
# =========================
for fname in os.listdir(INPUT_DIR):
    if not fname.lower().endswith(".wav"):
        continue

    path = os.path.join(INPUT_DIR, fname)
    audio, sr = librosa.load(path, sr=44100, mono=True)

    base = os.path.splitext(fname)[0]

    # ----- Window 1024 -----
    spec1024 = compute_log_norm_padded_spec(
        audio=audio, sr=sr, n_fft=1024, hop_length=512, max_frames=MAX_FRAMES_1024
    )
    # np.save(os.path.join(OUT_SPEC1024, f"{base}_spec1024.npy"), spec1024)

    # SPM for 1024: slightly stronger high-pass, plus a light kernel
    spm1024 = apply_spm_filters(
        spec1024,
        kernels=[KERNEL_STRONG, KERNEL_LIGHT],
        weights=[1.0, 0.5]
    )
    np.save(os.path.join(OUT_SPM1024, f"{base}_spec1024_spm.npy"), spm1024)

    # ----- Window 512 -----
    spec512 = compute_log_norm_padded_spec(
        audio=audio, sr=sr, n_fft=512, hop_length=256, max_frames=MAX_FRAMES_512
    )
    # np.save(os.path.join(OUT_SPEC512, f"{base}_spec512.npy"), spec512)

    # SPM for 512: lighter filtering (good for finer time resolution)
    spm512 = apply_spm_filters(
        spec512,
        kernels=[KERNEL_LIGHT],
        weights=[1.0]
    )
    np.save(os.path.join(OUT_SPM512, f"{base}_spec512_spm.npy"), spm512)

    print(
        f"{fname} -> "
        f"1024: raw {spec1024.shape}, spm {spm1024.shape} | "
        f"512: raw {spec512.shape}, spm {spm512.shape}"
    )


Streaming output truncated to the last 5000 lines.
English5539_standardized.wav -> 1024: raw (513, 87), spm (513, 87) | 512: raw (257, 173), spm (257, 173)
English5209_standardized.wav -> 1024: raw (513, 87), spm (513, 87) | 512: raw (257, 173), spm (257, 173)
English5502_standardized.wav -> 1024: raw (513, 87), spm (513, 87) | 512: raw (257, 173), spm (257, 173)
English5274_standardized.wav -> 1024: raw (513, 87), spm (513, 87) | 512: raw (257, 173), spm (257, 173)
English5314_standardized.wav -> 1024: raw (513, 87), spm (513, 87) | 512: raw (257, 173), spm (257, 173)
English5410_standardized.wav -> 1024: raw (513, 87), spm (513, 87) | 512: raw (257, 173), spm (257, 173)
English5186_standardized.wav -> 1024: raw (513, 87), spm (513, 87) | 512: raw (257, 173), spm (257, 173)
English5086_standardized.wav -> 1024: raw (513, 87), spm (513, 87) | 512: raw (257, 173), spm (257, 173)
English5465_standardized.wav -> 1024: raw (513, 87), spm (513, 87) | 512: raw (257, 173), spm (257, 173)
Engl

In [ ]:
!git clone https://github.com/sarah2005-cyber/auspex.git


Cloning into 'auspex'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 56 (delta 9), reused 54 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 17.33 MiB | 34.93 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [ ]:
!mv "/content/auspex/backend/Preprocessing Steps.ipynb" /content/auspex/


In [ ]:
!ls /content/auspex


 AttentionFeatureFusionSPM.py   LightweightCNNClassifier.py
 backend		       'Preprocessing Steps.ipynb'
 EmbeddingRateEstimator.py      README.md
 frontend		        ResidualUnit.py
 FullPipelineModel


In [ ]:
%cd /content/auspex
!git status

/content/auspex
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	Preprocessing Steps.ipynb

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
!git add "Preprocessing Steps.ipynb"
!git commit -m "Add preprocessing notebook"
!git push


[main 97fc907] Add preprocessing notebook
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 Preprocessing Steps.ipynb
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
!git config --global user.email "maryamsarah188@gmail.com"
!git config --global user.name "sarah2005-cyber"

In [ ]:
!git status

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [10]:
!git add "Preprocessing steps.ipynb"


fatal: pathspec 'Preprocessing steps.ipynb' did not match any files
